In [1]:
import pandas as pd
import pyarrow.parquet as pq

path = "../data/food.parquet"
parquet_file = pq.ParquetFile(path)

number_rows = parquet_file.metadata.num_rows
number_cols = parquet_file.metadata.num_columns

print(f"Le fichier se compose de {number_rows:,} lignes et de {number_cols} colonnes.")

schema_dataframe = pd.DataFrame({
    "column": parquet_file.schema_arrow.names,
    "type": [str(t) for t in parquet_file.schema_arrow.types]
})

# Display column data types
schema_dataframe

Le fichier se compose de 4,724,424 lignes et de 145 colonnes.


,column,type
0,additives_n,int32
1,additives_tags,list<element: string>
2,allergens_tags,list<element: string>
3,brands_tags,list<element: string>
4,brands,string
...,...,...
106,unknown_nutrients_tags,list<element: string>
107,vitamins_tags,list<element: string>
108,with_non_nutritive_sweeteners,int32
109,with_sweeteners,int32


### Affichage des ressources consommées en mémoire

In [3]:
# Load one row group as a real pandas DataFrame to measure actual memory usage
first_batch = next(parquet_file.iter_batches(batch_size=200_000))
datafram_sample = first_batch.to_pandas()

# memory_usage(deep=True) accounts for the real size of object dtypes (strings, lists),
# not just a fixed size per column — critical here since most OFF columns are text/tags
memory_sample_mb = datafram_sample.memory_usage(deep=True).sum() / (1024**2)
memory_per_row = datafram_sample.memory_usage(deep=True).sum() / len(datafram_sample)

# Extrapolate from the sample to the full file to estimate RAM needed for a full load
memory_estimate_gb = (memory_per_row * number_rows) / (1024**3)

# Note: this estimate will be well above the 7GB on-disk file size, since Parquet is
# compressed and pandas stores strings far less efficiently — this is the justification
# for chunked loading instead of pd.read_parquet() on the full file
print(f"Échantillon de {len(datafram_sample):,} lignes : {memory_sample_mb:.1f} Mo en mémoire")
print(f"Mémoire estimée pour les {number_rows:,} lignes du fichier complet : {memory_estimate_gb:.1f} Go")

Échantillon de 200,000 lignes : 2509.2 Mo en mémoire
Mémoire estimée pour les 4,724,424 lignes du fichier complet : 57.9 Go


### Affichage du taux de remplissage par colonne

In [4]:
# Compute fill rate (non-null ratio) for each column, processed in chunks
null_counts = pd.Series(0, index=parquet_file.schema_arrow.names)
rows_processed = 0

for batch in parquet_file.iter_batches(batch_size=200_000):
    dataframe_chunk = batch.to_pandas()
    rows_processed += len(dataframe_chunk)
    null_counts += dataframe_chunk.isna().sum()

fill_rate = (1 - null_counts / rows_processed).sort_values(ascending=False)
fill_rate_df = fill_rate.rename("fill_rate").to_frame()
fill_rate_df["fill_rate_pct"] = (fill_rate_df["fill_rate"] * 100).round(1)

fill_rate_df

,fill_rate,fill_rate_pct
code,1.000000,100.0
ingredients_text,1.000000,100.0
images,1.000000,100.0
generic_name,1.000000,100.0
product_name,1.000000,100.0
...,...,...
editors,0.012211,1.2
with_non_nutritive_sweeteners,0.006717,0.7
with_sweeteners,0.001019,0.1
photographers,0.000940,0.1


### Combien de produits vendus en France ? Quelle part nutriscore renseignée ? 
### Top 10 des marques - Taux de manquants sur les nutriments

In [ ]:
import pandas as pd
from collections import Counter

columns_to_load = [
    "countries_tags",
    "nutriscore_grade",
    "brands",
    "nutriments",
]

valid_nutriscore_grades = {"a", "b", "c", "d", "e"}
energy_synonym_names = {"energy-kcal", "energy", "energy-kj"}
other_key_nutrient_names = {"sugars_100g": "sugars", "salt_100g": "salt"}


def get_nutrient_value(nutrient_list, nutrient_name):
    """Extrait la valeur '100g' d'un nutriment donné dans la liste de dicts nutriments."""
    if nutrient_list is None:
        return None
    for nutrient_entry in nutrient_list:
        if nutrient_entry.get("name") == nutrient_name:
            return nutrient_entry.get("100g")
    return None


def has_energy_value(nutrient_list):
    """Vérifie si une valeur d'énergie est renseignée, sous n'importe lequel de ses noms connus."""
    if nutrient_list is None:
        return False
    for nutrient_entry in nutrient_list:
        if nutrient_entry.get("name") in energy_synonym_names:
            if nutrient_entry.get("100g") is not None:
                return True
    return False


# accumulateurs, mis à jour à chaque batch
rows_processed = 0
products_in_france_count = 0
products_with_nutriscore_in_france_count = 0
brand_counter_france = Counter()
missing_nutrient_counts = pd.Series(0, index=["energy_100g", "sugars_100g", "salt_100g"])

for batch in parquet_file.iter_batches(batch_size=200_000, columns=columns_to_load):
    dataframe_chunk = batch.to_pandas()
    rows_processed += len(dataframe_chunk)

    # Q1 : produits vendus en France
    is_sold_in_france_mask = dataframe_chunk["countries_tags"].apply(
        lambda tags: tags is not None and "en:france" in tags
    )
    products_in_france_count += is_sold_in_france_mask.sum()
    france_chunk = dataframe_chunk.loc[is_sold_in_france_mask]

    # Q2 : part avec Nutri-Score renseigné (sur le sous-ensemble France)
    products_with_nutriscore_in_france_count += france_chunk["nutriscore_grade"].isin(valid_nutriscore_grades).sum()

    # Q3 : marques les plus présentes (sur le sous-ensemble France)
    brands_in_chunk = france_chunk["brands"].dropna().str.split(",").explode().str.strip()
    brands_in_chunk = brands_in_chunk[brands_in_chunk != ""]
    brand_counter_france.update(brands_in_chunk)

    # Q4 : taux de manquants sur les nutriments clés
    energy_present_mask = dataframe_chunk["nutriments"].apply(has_energy_value)
    missing_nutrient_counts["energy_100g"] += (~energy_present_mask).sum()

    for column_label, nutrient_name in other_key_nutrient_names.items():
        extracted_values = dataframe_chunk["nutriments"].apply(get_nutrient_value, nutrient_name=nutrient_name)
        missing_nutrient_counts[column_label] += extracted_values.isna().sum()

# --- résultats, calculés une seule fois à la fin ---

nutriscore_fill_rate_france_pct = (products_with_nutriscore_in_france_count / products_in_france_count) * 100
top_ten_brands_france = pd.Series(dict(brand_counter_france.most_common(10)), name="product_count")
missing_nutrient_rate_pct = (missing_nutrient_counts / rows_processed * 100).round(2)

print(f"Produits vendus en France : {products_in_france_count:,}")
print(f"Part avec Nutri-Score renseigné (France) : {nutriscore_fill_rate_france_pct:.2f} %")
print("\nTop 10 marques (France) :")
print(top_ten_brands_france)
print("\nTaux de manquants sur les nutriments clés :")
print(missing_nutrient_rate_pct)

Produits vendus en France : 1,259,443
Part avec Nutri-Score renseigné (France) : 37.12 %

Top 10 marques (France) :
U                16107
Carrefour        15126
Auchan            8200
Marque Repère     7674
Casino            6465
Nestlé            5944
Leader Price      5694
Lidl              4676
Monoprix          4481
Cora              4119
Name: product_count, dtype: int64

Taux de manquants sur les nutriments clés :
energy_100g    27.71
sugars_100g    32.45
salt_100g      37.25
dtype: float64
